# National Electricity Grid Network Analysis
### CS 112 Final Project — Data Engineering & Network Analysis

**Author:** Clinton
**Dataset:** Synthetic grid data grounded in Ghana's real utilities (ECG, NEDCo, GRIDCo, VRA) and the West African Power Pool (WAPP)

This notebook consolidates the full project pipeline into a single, runnable file, in the order specified by the project brief:

1. Dataset generation
2. Task 1 — Load and inspect
3. Task 2 — Data cleaning
4. Task 3 — Exploratory data analysis (EDA)
5. Task 4 — Data integration (merging)
6. Task 5 — Network analysis & visualization
7. Geospatial analysis (folium)
8. Business intelligence & reliability analysis
9. Interactive Streamlit dashboard (reference code — run separately, see note below)

> **Note on running this notebook:** cells assume a working directory of the project root (the folder containing `data/`, `output/`, and this notebook), matching how the original scripts read `data/utilities.csv` etc. Run `GenerateData.py`'s cell first if the `data/` folder is not already populated.


## 0. Setup

Install/import dependencies used across the notebook.

In [ ]:
# If running fresh, install the required libraries:
# %pip install pandas numpy matplotlib networkx folium geopy scikit-learn plotly streamlit streamlit-folium

import os
os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)


## 1. Dataset Generation

Seeded (`random.seed(42)`) synthetic generator producing `utilities.csv`, `substations.csv`, and `lines.csv`, grounded in real Ghanaian geography and WAPP cross-border interconnections.

In [ ]:
"""
Synthetic dataset generator for the National Electricity Grid Network Analysis project.
Produces three CSVs in the same spirit as the OpenFlights airlines/airports/routes trio:
    utilities.csv    (like airlines.csv)
    substations.csv  (like airports.csv)
    lines.csv        (like routes.csv)
 
Grounded in Ghana's grid (ECG, NEDCo, GRIDCo, VRA) with cross-border interconnections
reflecting the real West African Power Pool (WAPP).
"""
import csv
import math
import random
 
random.seed(42)
 
# ---------------------------------------------------------------------------
# Utilities (analogous to airlines.csv)
# ---------------------------------------------------------------------------
utilities = [
    # utility_id, name, alias, code, type, country, active
    (1, "Electricity Company of Ghana", "ECG", "ECG", "Distribution", "Ghana", "Y"),
    (2, "Northern Electricity Distribution Company", "NEDCo", "NED", "Distribution", "Ghana", "Y"),
    (3, "Ghana Grid Company", "GRIDCo", "GRD", "Transmission", "Ghana", "Y"),
    (4, "Volta River Authority", "VRA", "VRA", "Generation", "Ghana", "Y"),
    (5, "Compagnie Ivoirienne d'Electricite", "CIE", "CIE", "Distribution", "Cote d'Ivoire", "Y"),
    (6, "Communaute Electrique du Benin", "CEB", "CEB", "Transmission", "Togo/Benin", "Y"),
    (7, "Societe Beninoise d'Energie Electrique", "SBEE", "SBE", "Distribution", "Benin", "Y"),
    (8, "Electricite de Guinee", "EDG", "EDG", "Distribution", "Guinea", "N"),
    (9, "Sonabel", "SONABEL", "SNB", "Distribution", "Burkina Faso", "Y"),
    (10, "Enclave Power Company", "EPC", "EPC", "Generation", "Ghana", "N"),
]
 
# ---------------------------------------------------------------------------
# Substations (analogous to airports.csv) — coordinates are approximate/illustrative
# ---------------------------------------------------------------------------
ghana_regions = {
    "Greater Accra": [("Achimota", 5.614, -0.224), ("Tema", 5.669, -0.017),
                       ("Mallam", 5.560, -0.298), ("Legon", 5.650, -0.186),
                       ("Kaneshie", 5.560, -0.238), ("Aboadze Junction", 5.590, -0.150)],
    "Ashanti": [("Kumasi Central", 6.688, -1.624), ("Ejisu", 6.719, -1.463),
                ("Obuasi", 6.202, -1.663), ("Mampong", 7.062, -1.400), ("Konongo", 6.618, -1.219)],
    "Western": [("Takoradi", 4.895, -1.759), ("Aboadze", 4.985, -1.766),
                ("Tarkwa", 5.301, -1.994), ("Axim", 4.867, -2.241)],
    "Central": [("Cape Coast", 5.106, -1.246), ("Winneba", 5.352, -0.622),
                ("Kasoa", 5.533, -0.416), ("Assin Fosu", 5.699, -1.492)],
    "Eastern": [("Koforidua", 6.094, -0.259), ("Akosombo", 6.300, 0.055),
                ("Nkawkaw", 6.550, -0.767), ("Suhum", 6.041, -0.451)],
    "Volta": [("Ho", 6.611, 0.471), ("Kpong", 6.150, 0.100),
              ("Hohoe", 7.152, 0.472), ("Sogakope", 6.007, 0.573)],
    "Bono": [("Sunyani", 7.339, -2.326), ("Techiman", 7.590, -1.938), ("Berekum", 7.453, -2.585)],
    "Northern": [("Tamale", 9.403, -0.842), ("Yendi", 9.442, -0.011), ("Savelugu", 9.625, -0.826)],
    "Upper East": [("Bolgatanga", 10.787, -0.851), ("Bawku", 11.058, -0.243)],
    "Upper West": [("Wa", 10.061, -2.501)],
}
 
cross_border = [
    ("Bolgatanga Interconnection", "Burkina Faso border", 11.20, -0.75),
    ("Elubo Border Station", "Cote d'Ivoire border", 5.20, -2.85),
    ("Aflao Border Station", "Togo border", 6.12, 1.19),
    ("Lome Transmission Hub", "Togo", 6.13, 1.22),
    ("Cotonou Transmission Hub", "Benin", 6.37, 2.43),
    ("Abidjan Transmission Hub", "Cote d'Ivoire", 5.35, -4.00),
    ("Bobo-Dioulasso Hub", "Burkina Faso", 11.18, -4.30),
    ("Conakry Transmission Hub", "Guinea", 9.64, -13.58),
]
 
voltage_levels = [11, 33, 69, 161, 330]
 
substations = []
sid = 1
name_to_id = {}
for region, places in ghana_regions.items():
    for name, lat, lon in places:
        voltage = random.choice(voltage_levels)
        sub_type = ("Transmission" if voltage >= 161 else
                    "Bulk Supply Point" if voltage == 69 else "Distribution")
        capacity = round(random.uniform(15, 400) if sub_type != "Distribution"
                          else random.uniform(5, 60), 1)
        status = "Active" if random.random() > 0.05 else "Inactive"
        substations.append([
            sid, f"{name} Substation", name, region, "Ghana",
            round(lat + random.uniform(-0.01, 0.01), 4), round(lon + random.uniform(-0.01, 0.01), 4),
            voltage, capacity, random.randint(1965, 2023), sub_type, status,
        ])
        name_to_id[name] = sid
        sid += 1
 
for name, country, lat, lon in cross_border:
    voltage = random.choice([161, 330])
    capacity = round(random.uniform(100, 500), 1)
    substations.append([
        sid, f"{name}", name, country, country.split()[0],
        round(lat, 4), round(lon, 4), voltage, capacity,
        random.randint(1980, 2020), "Transmission", "Active",
    ])
    name_to_id[name] = sid
    sid += 1
 
# ---------------------------------------------------------------------------
# Transmission/Distribution lines (analogous to routes.csv)
# ---------------------------------------------------------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))
 
sub_by_id = {row[0]: row for row in substations}
 
lines = []
lid = 1
seen_pairs = set()
 
# Connect substations within each region into a loosely meshed network
for region, places in ghana_regions.items():
    ids_in_region = [name_to_id[n] for n, _, _ in places]
    for i, a in enumerate(ids_in_region):
        for b in ids_in_region[i + 1:]:
            if random.random() < 0.55:
                pair = tuple(sorted((a, b)))
                if pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                utility_id = random.choice([1, 2, 3])
                sub_a, sub_b = sub_by_id[a], sub_by_id[b]
                dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6])
                             * random.uniform(1.05, 1.3), 1)
                voltage = min(sub_a[7], sub_b[7])
                lines.append([
                    lid, utility_id, a, sub_a[1], b, sub_b[1],
                    voltage, dist, round(random.uniform(20, 300), 1),
                    "Active" if random.random() > 0.08 else "Under Maintenance",
                    random.choice(["Overhead", "Underground"]),
                ])
                lid += 1
 
# A handful of inter-regional backbone lines (transmission-level, GRIDCo)
region_hub = {r: name_to_id[places[0][0]] for r, places in ghana_regions.items()}
region_names = list(region_hub.keys())
for i in range(len(region_names) - 1):
    a = region_hub[region_names[i]]
    b = region_hub[region_names[i + 1]]
    sub_a, sub_b = sub_by_id[a], sub_by_id[b]
    dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * 1.15, 1)
    lines.append([
        lid, 3, a, sub_a[1], b, sub_b[1], 330, dist,
        round(random.uniform(200, 600), 1), "Active", "Overhead",
    ])
    lid += 1
 
# Cross-border interconnections (WAPP-style)
border_links = [
    ("Bolgatanga", "Bolgatanga Interconnection", 9),
    ("Bolgatanga Interconnection", "Bobo-Dioulasso Hub", 9),
    ("Elubo Border Station", "Kumasi Central", 5),
    ("Elubo Border Station", "Abidjan Transmission Hub", 5),
    ("Aflao Border Station", "Tema", 6),
    ("Aflao Border Station", "Lome Transmission Hub", 6),
    ("Lome Transmission Hub", "Cotonou Transmission Hub", 6),
]
for src_name, dst_name, utility_id in border_links:
    if src_name not in name_to_id or dst_name not in name_to_id:
        continue
    a, b = name_to_id[src_name], name_to_id[dst_name]
    sub_a, sub_b = sub_by_id[a], sub_by_id[b]
    dist = round(haversine_km(sub_a[5], sub_a[6], sub_b[5], sub_b[6]) * 1.1, 1)
    lines.append([
        lid, utility_id, a, sub_a[1], b, sub_b[1], 330, dist,
        round(random.uniform(150, 400), 1), "Active", "Overhead",
    ])
    lid += 1
 
# ---------------------------------------------------------------------------
# Write CSVs
# ---------------------------------------------------------------------------
with open("data/utilities.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Utility ID", "Name", "Alias", "Code", "Type", "Country", "Active"])
    w.writerows(utilities)
 
with open("data/substations.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Substation ID", "Name", "Short Name", "Region", "Country", "Latitude", "Longitude",
                "Voltage (kV)", "Capacity (MVA)", "Commissioning Year", "Type", "Status"])
    w.writerows(substations)
 
with open("data/lines.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Line ID", "Utility ID", "Source Substation ID", "Source Substation",
                "Destination Substation ID", "Destination Substation", "Voltage (kV)",
                "Length (km)", "Capacity (MVA)", "Status", "Line Type"])
    w.writerows(lines)
 
print(f"utilities: {len(utilities)} rows")
print(f"substations: {len(substations)} rows")
print(f"lines: {len(lines)} rows")


## 2. Task 1 — Load and Inspect the Datasets

Load the three CSVs and inspect structure, types, and completeness.

In [ ]:
# Step 1: Load and examine raw data
import pandas as pd

utilities=pd.read_csv("data/utilities.csv")
substations=pd.read_csv("data/substations.csv")
lines=pd.read_csv("data/lines.csv")  

print(utilities.info())
print(utilities.head())
print("-----------------")
print(substations.info())
print(substations.head())
print("-----------------")
print(lines.info())
print(lines.head())
print("-----------------")
print(utilities.isnull().sum())


## 3. Task 2 — Data Cleaning and Validation

Validate foreign keys (Source/Destination Substation IDs, Utility IDs), enforce numeric types, check coordinate bounds, and drop duplicates.

In [ ]:
import pandas as pd

utilities=pd.read_csv("data/utilities.csv")
substations=pd.read_csv("data/substations.csv")
lines=pd.read_csv("data/lines.csv")

# Step 2: Handle missing values
# Even though the generator produces clean data, treat this step seriously —
# real grid asset registers always have gaps. Decide on imputation strategies
# for different columns and document your decisions and rationale.



# Step 3: Data validation
# Verify every Source/Destination Substation ID in lines.csv exists in substations.csv
checkValS=set(substations['Substation ID'])
checkValU=set(utilities['Utility ID'])

invalidSources = lines[~lines['Source Substation ID'].isin(checkValS)]
invalidDestinations = lines[~lines['Destination Substation ID'].isin(checkValS)]
invalidUtilites=lines[~lines['Utility ID'].isin(checkValU)]

print(f"Missing Source Substations: {len(invalidSources)}")
print(f"missing Destination Substations: {len(invalidDestinations)}")
print(f"Lines with Invalid Utility IDs: {len(invalidUtilites)}")
print(f"----------------------------------------")

# Ensure data type consistency (numeric columns are truly numeric)

substations['Latitude'] = pd.to_numeric(substations['Latitude'], errors='coerce')
substations['Longitude'] = pd.to_numeric(substations['Longitude'], errors='coerce')
substations['Capacity (MVA)'] = pd.to_numeric(substations['Capacity (MVA)'], errors='coerce')
lines['Length (km)'] = pd.to_numeric(lines['Length (km)'], errors='coerce')


# Validate that latitude/longitude fall within plausible West African bounds
invalidLats = substations[(substations['Latitude'] < 4.5) | (substations['Latitude'] > 11.5)]
invalidLongs = substations[(substations['Longitude'] < -4.5) | (substations['Longitude'] > 3.0)]

print(f"Substations with out-of-bounds Latitude: {len(invalidLats)}")
print(f"Substations with out-of-bounds Longitude: {len(invalidLongs)}")
print(f"----------------------------------------")

# Check for duplicate entries
 
print(f"Duplicate Rows in Utilities:{utilities.duplicated().sum()}")
print(f"Duplicate Rows in Substations:{substations.duplicated().sum()}")
print(f"Duplicate Rows in lines:{lines.duplicated().sum()}")

utilities=utilities.drop_duplicates()
substations=substations.drop_duplicates()
lines=lines.drop_duplicates()

# Step 2: Handle missing values
print("Missing Values in Utilities:")
print(utilities.isnull().sum(), "\n")
print("Missing Values in Substations:")
print(substations.isnull().sum(), "\n")
print("Missing Values in Lines:")
print(lines.isnull().sum(), "\n")
print("----------------------------------------")

## 4. Task 3 — Exploratory Data Analysis (EDA)

Descriptive statistics, frequency distributions, most-connected substations, and regional breakdowns. Saves `output/eda_regions.png` and `output/eda_top_substations.png`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

utilities=pd.read_csv("data/utilities.csv")
substations=pd.read_csv("data/substations.csv")
lines=pd.read_csv("data/lines.csv")     

print(substations[['Latitude','Longitude','Voltage (kV)','Capacity (MVA)','Commissioning Year']].describe())
print("-----------------------------------------------------------------------------------")
print(lines[['Voltage (kV)','Length (km)','Capacity (MVA)']].describe())
print("-----------------------------------------------------------------------------------")

print('lines')
print(lines['Destination Substation'].value_counts())
print("-----------------------------------------------------------------------------------")
print(lines['Source Substation'].value_counts())
print("-----------------------------------------------------------------------------------")
print(lines['Status'].value_counts())
print("-----------------------------------------------------------------------------------")
print(lines['Line Type'].value_counts())
print("-----------------------------------------------------------------------------------")



print('Substations')
print(substations['Region'].value_counts())
print("-----------------------------------------------------------------------------------")
print(substations['Country'].value_counts())
print("-----------------------------------------------------------------------------------")
print(substations['Type'].value_counts())
print("-----------------------------------------------------------------------------------")
print(substations['Status'].value_counts())
print("-----------------------------------------------------------------------------------")

print(f'Utilities')
print(utilities['Country'].value_counts())
print("-----------------------------------------------------------------------------------")
print(utilities['Type'].value_counts())
print("-----------------------------------------------------------------------------------")
print(utilities['Active'].value_counts())
print("-----------------------------------------------------------------------------------")

print("Top Utilities")
topUtilities = lines['Utility ID'].value_counts().to_dict()
print(topUtilities)

sourceCounts = lines['Source Substation'].value_counts()
destCounts = lines['Destination Substation'].value_counts()

# Add them together to get total connected lines per substation

totalConnections = sourceCounts.add(destCounts, fill_value=0).sort_values(ascending=False)

print("Top 10 Most-Connected Substations")
print(totalConnections.head(10))

# Distribution of substations by region
substations_per_region = substations['Region'].value_counts()
print("Substations per Region")
print(substations_per_region)

# Total line capacity or length by region 
# If lines are linked to substations, merge lines with substations first:
linesWithRegion = lines.merge(
    substations[['Substation ID', 'Region']], 
    left_on='Source Substation ID', 
    right_on='Substation ID', 
    how='left'
)

region_line_length = linesWithRegion.groupby('Region')['Length (km)'].sum().sort_values(ascending=False)
print("Total Transmission Line Length (km) by Region")
print(region_line_length)

# Individual Region distribution
print("Substation Region Distribution")
print(substations['Region'].value_counts())

# Individual status distribution
print("Substation Status Distribution")
print(substations['Status'].value_counts())

# Individual voltage level distribution     
print("Substation Voltage Distribution")
print(substations['Voltage (kV)'].value_counts())

# Cross-tabulation: Voltage levels broken down by Active vs. Inactive status
voltage_by_status = pd.crosstab(substations['Voltage (kV)'], substations['Status'])
print("Voltage Levels by Operational Status")
print(voltage_by_status)


plt.figure(figsize=(10, 6))
substations['Region'].value_counts().plot(kind='bar', title='Substations by Region')
plt.tight_layout()
plt.savefig('output/eda_regions.png')
plt.show()

plt.figure(figsize=(10, 6))
totalConnections.head(10).plot(kind='bar', title='Top 10 Most-Connected Substations')
plt.tight_layout()
plt.savefig('output/eda_top_substations.png')
plt.show()



# - Executive summary with key metrics   
# - Interactive map with filtering options (region, voltage, utility)
# - Network analysis visualization
# - Business intelligence / reliability charts
# - Search functionality for specific substations/lines
# - Comparison tools for different utilities
# Create publication-quality visualizations
# - Animated maps showing grid expansion by Commissioning Year
# - 3D network visualizations
# - Interactive chord diagrams for inter-regional power-line flows
# - Heatmaps for line density and maintenance-status concentration
# - Comparative charts for utility infrastructure footprints


## 5. Task 4 — Data Integration (Merging)

Join utilities and substation region/status data onto the lines table to build a master dataset (`data/masterDataset.csv`), and build lookup dictionaries for fast querying.

In [ ]:
import pandas as pd

lines = pd.read_csv('data/lines.csv')   
substations = pd.read_csv('data/substations.csv')
utilities = pd.read_csv('data/utilities.csv')
# 1. Convert all ID columns to strings to prevent data type mismatches
lines['Utility ID'] = lines['Utility ID'].astype(str)
lines['Source Substation ID'] = lines['Source Substation ID'].astype(str)
lines['Destination Substation ID'] = lines['Destination Substation ID'].astype(str)

substations['Substation ID'] = substations['Substation ID'].astype(str)
utilities['Utility ID'] = utilities['Utility ID'].astype(str)

# 2. Join Utilities onto Lines
masterDf = lines.merge(utilities, on='Utility ID', how='left')

masterDf = masterDf.merge(
    substations[['Substation ID', 'Region']],
    left_on='Source Substation ID',
    right_on='Substation ID',
    how='left'
).rename(columns={'Region': 'Source Region'}).drop(columns=['Substation ID'])

masterDf = masterDf.merge(
    substations[['Substation ID', 'Region']],
    left_on='Destination Substation ID',
    right_on='Substation ID',
    how='left'
).rename(columns={'Region': 'Destination Region'}).drop(columns=['Substation ID'])

region_lookup = substations.set_index('Substation ID')['Region'].to_dict()

print(masterDf.columns.tolist())
print(masterDf[['Source Substation ID', 'Source Region', 'Destination Substation ID', 'Destination Region']].head())

# Dictionary 2: Quick lookup of Substation Status by Substation ID
status_lookup = substations.set_index('Substation ID')['Status'].to_dict()

print(f"Rows with missing Source Region: {masterDf['Source Region'].isna().sum()}")
print(f"Rows with missing Destination Region: {masterDf['Destination Region'].isna().sum()}")

print(f"\n--- Lookup Dictionaries Created ---")
print(f"Region lookup sample: {list(region_lookup.items())[:2]}")
print(f"Status lookup sample: {list(status_lookup.items())[:2]}")


masterDf.to_csv('data/masterDataset.csv', index=False)
print("Master dataset successfully saved to 'data/masterDataset.csv")

## 6. Task 5 — Network Analysis and Visualization

Model the grid as an undirected graph (power can flow either direction along a line). Computes:
- Degree, betweenness, closeness centrality, and PageRank (custom power-iteration implementation)
- Network diameter, average path length, clustering coefficient
- Community detection (greedy modularity)
- **N-1 contingency analysis**: articulation points (critical substations) and bridges (critical lines) — i.e. what happens if a key substation is removed
- Global/local efficiency
- A network topology visualization highlighting critical nodes/lines, saved to `output/network_graph.png`


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

linesDs= pd.read_csv('data/lines.csv')
substationsDs= pd.read_csv('data/substations.csv')
 

# Create network graph — undirected, since AC power can flow either way
# along a line depending on system conditions (unlike a scheduled flight,
# which always has a fixed origin and destination)
G = nx.Graph()
# Add substations as nodes with attributes (region, voltage, coordinates, etc.)
# Add Substation Nodes
def calc_pagerank(graph, alpha=0.85, max_iter=100, tol=1e-6):
    nodes = list(graph.nodes())
    n = len(nodes)
    if n == 0:
        return {}
    
    # Initialize uniform scores
    p = {node: 1.0 / n for node in nodes}
    
    # Power iteration
    for _ in range(max_iter):
        p_last = p.copy()
        p = {node: (1.0 - alpha) / n for node in nodes}
        
        for node in nodes:
            neighbors = list(graph.neighbors(node))
            if neighbors:
                share = alpha * p_last[node] / len(neighbors)
                for nbr in neighbors:
                    p[nbr] += share
            else:
                for nbr in nodes:
                    p[nbr] += alpha * p_last[node] / n
                    
        err = sum(abs(p[node] - p_last[node]) for node in nodes)
        if err < tol:
            break
            
    return p

for _, row in substationsDs.iterrows():
    G.add_node(
        row['Substation ID'],
        name=row.get('Name', f"Substation {row['Substation ID']}"),
        region=row.get('Region', 'N/A'),
        voltage=row.get('Voltage (kV)', 0)
    )

# Add Transmission Line Edges
for _, row in linesDs.iterrows():
    G.add_edge(
        row['Source Substation ID'],
        row['Destination Substation ID'],
        line_id=row['Line ID'],
        length=row.get('Length (km)', 1.0),
        capacity=row.get('Capacity (MVA)', 1.0)
    )

print(f"Graph Created: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Edges\n")
# Add lines as edges with weights (length, capacity, etc.)
 
# Calculate network metrics
degree_cent = nx.degree_centrality(G)   
betweenness_cent = nx.betweenness_centrality(G, weight='length')
closeness_cent = nx.closeness_centrality(G, distance='length')
pagerank_cent = calc_pagerank(G)

centrality_df = pd.DataFrame({
    'Node ID': list(G.nodes()),
    'Degree Centrality': [degree_cent[n] for n in G.nodes()],
    'Betweenness Centrality': [betweenness_cent[n] for n in G.nodes()],
    'Closeness Centrality': [closeness_cent[n] for n in G.nodes()],
    'PageRank': [pagerank_cent[n] for n in G.nodes()]
}).sort_values(by='Degree Centrality', ascending=False)

print("--- TOP CENTRAL SUBSTATIONS ---")
print(centrality_df.head(), "\n")
 
# STEP 3: COMPUTE NETWORK STRUCTURE METRICS

# Evaluate on largest connected component if disconnected
if nx.is_connected(G):
    target_graph = G
else:
    largest_cc = max(nx.connected_components(G), key=len)
    target_graph = G.subgraph(largest_cc)

diameter = nx.diameter(target_graph, weight='length')
avg_path_len = nx.average_shortest_path_length(target_graph, weight='length')
avg_clustering = nx.average_clustering(G)

print("--- GLOBAL STRUCTURE METRICS ---")
print(f"Network Diameter: {diameter:.2f} km")
print(f"Average Path Length: {avg_path_len:.2f} km")
print(f"Average Clustering Coefficient: {avg_clustering:.4f}\n")

# STEP 4: COMMUNITY DETECTION & VULNERABILITIES

# Detect sub-grid communities
communities = list(nx.community.greedy_modularity_communities(G))

# Find critical single points of failure

articulation_points = list(nx.articulation_points(G))  # Cut Nodes
bridge_lines = list(nx.bridges(G))                     # Bridge Edges

print("--- VULNERABILITY ANALYSIS ---")
print(f"Detected Sub-grid Communities: {len(communities)}")
print(f"Critical Substations (Cut Nodes): {articulation_points}")
print(f"Critical Transmission Lines (Bridges): {bridge_lines}\n")

# STEP 5: MEASURE NETWORK EFFICIENCY
glob_eff = nx.global_efficiency(G)
loc_eff = nx.local_efficiency(G)

print("--- NETWORK EFFICIENCY ---")
print(f"Global Efficiency: {glob_eff:.4f}")
print(f"Local Efficiency: {loc_eff:.4f}\n")



# STEP 6: VISUALIZE GRAPH

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42)

# Separate normal vs critical nodes
normal_nodes = [n for n in G.nodes() if n not in articulation_points]

# Draw Nodes
nx.draw_networkx_nodes(G, pos, nodelist=normal_nodes, node_color='blue', node_size=500, label='Substation')
nx.draw_networkx_nodes(G, pos, nodelist=articulation_points, node_color='red', node_size=700, label='Critical Cut Node')

# Separate normal vs bridge edges
normal_edges = [e for e in G.edges() if e not in bridge_lines and (e[1], e[0]) not in bridge_lines]

# Draw Edges
nx.draw_networkx_edges(G, pos, edgelist=normal_edges, edge_color='gray', width=1.5, label='Normal Line')
nx.draw_networkx_edges(G, pos, edgelist=bridge_lines, edge_color='orange', width=2.5, style='dashed', label='Bridge Line')

# Labels and Setup
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')
plt.title("Power Grid Network Topology & Vulnerability Analysis", fontsize=14, fontweight='bold')
plt.legend(scatterpoints=1, loc='upper left')
plt.axis('off')
plt.tight_layout()
plt.savefig('output/network_graph')
plt.show()



## 7. Geospatial Analysis

Recomputes exact line distances using geodesic (great-circle) calculations, categorizes lines by length, runs DBSCAN clustering on substation coordinates to find geographic clusters, and builds a multi-layer interactive Folium map (voltage-colored substations, transmission lines, density heatmap, geographic clusters). Saves `ghana_grid_spatial_analysis.html`.


In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster
from geopy.distance import geodesic
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

# ==========================================
# 1. LOAD DATA & PREPARE COORDINATES
# ==========================================
lines = pd.read_csv('data/lines.csv')
substations = pd.read_csv('data/substations.csv')
utilities = pd.read_csv('data/utilities.csv')

# Ensure IDs are strings
lines['Source Substation ID'] = lines['Source Substation ID'].astype(str)
lines['Destination Substation ID'] = lines['Destination Substation ID'].astype(str)
substations['Substation ID'] = substations['Substation ID'].astype(str)

# Create a coordinate dictionary for fast lookup: { 'ID': (lat, lon) }
coords_dict = substations.set_index('Substation ID')[['Latitude', 'Longitude']].apply(tuple, axis=1).to_dict()

# ==========================================
# 2. DISTANCE ANALYSIS & GEOMETRY
# ==========================================
def calculate_geodesic(row):
    src = row['Source Substation ID']
    dst = row['Destination Substation ID']
    if src in coords_dict and dst in coords_dict:
        # geodesic expects (lat, lon)
        return geodesic(coords_dict[src], coords_dict[dst]).kilometers
    return np.nan

# Recompute exact distances
lines['Calculated Length (km)'] = lines.apply(calculate_geodesic, axis=1)

# Categorize Lines
def categorize_distance(dist):
    if pd.isna(dist): return 'Unknown'
    if dist < 50: return 'Short (<50km)'
    elif dist <= 150: return 'Medium (50-150km)'
    else: return 'Long (>150km)'

lines['Distance Category'] = lines['Calculated Length (km)'].apply(categorize_distance)

print("--- DISTANCE DISTRIBUTION ANALYSIS ---")
print(lines['Distance Category'].value_counts(), "\n")

# ==========================================
# 3. SUBSTATION CLUSTERING (DBSCAN)
# ==========================================
# Find geographic clusters of substations (e.g., max 30km apart to be in the same cluster)
coords_array = substations[['Latitude', 'Longitude']].dropna().values
# Convert 30km to radians for haversine metric (Earth radius approx 6371 km)
epsilon = 30 / 6371.0 

dbscan = DBSCAN(eps=epsilon, min_samples=3, algorithm='ball_tree', metric='haversine')
# Fit model on radians
substations['Cluster ID'] = dbscan.fit_predict(np.radians(coords_array))

print("--- GEOGRAPHIC CLUSTERING ---")
print(f"Number of distinct high-density clusters found: {len(set(substations['Cluster ID'])) - 1}")
print(f"Number of isolated substations (noise): {list(substations['Cluster ID']).count(-1)}\n")

# ==========================================
# 4. BUILD INTERACTIVE MULTI-LAYER MAP
# ==========================================
# Center map on Ghana (approx 7.9465 N, 1.0232 W)
ghana_map = folium.Map(location=[7.9465, -1.0232], zoom_start=6, tiles='CartoDB Positron')

# Define Map Layers
layer_substations = folium.FeatureGroup(name='Substations (by Voltage)')
layer_lines = folium.FeatureGroup(name='Transmission Lines')
layer_heatmap = folium.FeatureGroup(name='Substation Density Heatmap')
layer_clusters = folium.FeatureGroup(name='Geographic Clusters')

# --- A. Substation Density Heatmap ---
heat_data = [[row['Latitude'], row['Longitude']] for index, row in substations.dropna(subset=['Latitude', 'Longitude']).iterrows()]
HeatMap(heat_data, radius=15, blur=20).add_to(layer_heatmap)

# --- B. Substations by Voltage ---
voltage_colors = {11: 'purple',33: 'green', 69: 'blue', 161: 'orange', 330: 'red'}

for _, row in substations.dropna(subset=['Latitude', 'Longitude']).iterrows():
    v = row.get('Voltage (kV)', 0)
    color = voltage_colors.get(v, 'gray') # Default to gray if unknown
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5 + (v / 50), # Scale radius slightly by voltage
        color=color,
        fill=True,
        fill_opacity=0.7,
        tooltip=f"{row.get('Name', 'Unknown')} ({v}kV) - {row.get('Region', '')}"
    ).add_to(layer_substations)

# --- C. Transmission Lines ---
for _, row in lines.dropna(subset=['Calculated Length (km)']).iterrows():
    src = row['Source Substation ID']
    dst = row['Destination Substation ID']
    
    if src in coords_dict and dst in coords_dict:
        folium.PolyLine(
            locations=[coords_dict[src], coords_dict[dst]],
            weight=2,
            color='black' if row['Distance Category'] != 'Long (>150km)' else 'purple',
            opacity=0.6,
            tooltip=f"Line: {row.get('Line ID', '')} | Length: {row['Calculated Length (km)']:.1f}km"
        ).add_to(layer_lines)

# --- D. Geographic Clusters ---
# Only plot nodes that belong to a valid cluster (ID >= 0)
cluster_colors = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4', '#46f0f0']
for _, row in substations[substations['Cluster ID'] >= 0].iterrows():
    c_id = int(row['Cluster ID'])
    c_color = cluster_colors[c_id % len(cluster_colors)]
    
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color='lightgray', icon_color=c_color, icon='bolt', prefix='fa'),
        tooltip=f"Cluster {c_id}"
    ).add_to(layer_clusters)

# Add layers to map and include Layer Control
layer_heatmap.add_to(ghana_map)
layer_lines.add_to(ghana_map)
layer_substations.add_to(ghana_map)
layer_clusters.add_to(ghana_map)

folium.LayerControl().add_to(ghana_map)

# Save output
ghana_map.save("ghana_grid_spatial_analysis.html")
print("Map successfully saved as 'ghana_grid_spatial_analysis.html'. Open this file in your web browser!")

## 8. Business Intelligence & Reliability Analysis

Asset age and fault-risk flags, capacity-utilization / upgrade-candidate flags, a technical line-loss proxy, maintenance-rate by utility, a capacity-concentration (HHI) risk index, and underserved-region identification. Builds a 4-panel interactive BI dashboard, saved to `grid_business_intelligence_dashboard.html`.


In [ ]:
import pandas as pd
import numpy as np
import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. LOAD & PREPARE DATASETS
# ==========================================
lines = pd.read_csv('data/lines.csv')
substations = pd.read_csv('data/substations.csv')
utilities = pd.read_csv('data/utilities.csv')

current_year = datetime.datetime.now().year

# Clean String IDs
lines['Source Substation ID'] = lines['Source Substation ID'].astype(str)
lines['Destination Substation ID'] = lines['Destination Substation ID'].astype(str)
lines['Utility ID'] = lines['Utility ID'].astype(str)
substations['Substation ID'] = substations['Substation ID'].astype(str)
utilities['Utility ID'] = utilities['Utility ID'].astype(str)

# Merge Utility Names onto lines only
lines = lines.merge(utilities[['Utility ID', 'Name']], on='Utility ID', how='left')
substations['Name'] = 'National Grid Operator'

# ==========================================
# 2. LOAD, CAPACITY & AGE ANALYSIS
# ==========================================
# A. Asset Age Calculation
substations['Asset Age'] = current_year - substations.get('Commissioning Year', current_year)
substations['High Fault Risk'] = substations['Asset Age'] > 30 # Flag assets > 30 years old

# B. Capacity Utilization & Upgrade Candidates
# Substation capacity load proxy (e.g., >85% rated capacity flagged as upgrade candidate)
if 'Peak Load (MW)' in substations.columns and 'Capacity (MW)' in substations.columns:
    substations['Utilization (%)'] = (substations['Peak Load (MW)'] / substations['Capacity (MW)']) * 100
else:
    # Proxy utilization if exact peak load column is absent
    substations['Utilization (%)'] = np.random.uniform(50, 95, size=len(substations))

substations['Upgrade Candidate'] = substations['Utilization (%)'] > 85

# C. Technical Loss Proxy Analysis
# Technical Loss Proxy Formula: Loss ~ (Length * Power Flow) / (Voltage^2)
def calc_loss_proxy(row):
    length = row.get('Length (km)', 10)
    voltage = row.get('Voltage (kV)', 161)
    capacity = row.get('Capacity (MW)', 100)
    if voltage <= 0: voltage = 161
    return (length * capacity) / (voltage ** 2)

lines['Loss Proxy Index'] = lines.apply(calc_loss_proxy, axis=1)

# ==========================================
# 3. RELIABILITY & RISK METRICS
# ==========================================
# Maintenance Status Rates by Utility & Region
lines['Is Maintenance'] = lines.get('Status', 'Active').astype(str).str.contains('Maintenance', case=False)
maint_by_utility = lines.groupby('Name')['Is Maintenance'].mean() * 100

# Capacity Concentration Risk (Herfindahl-Hirschman Index - HHI Proxy)
total_grid_capacity = substations['Capacity (MW)'].sum() if 'Capacity (MW)' in substations.columns else len(substations)
substations['Capacity Share'] = (substations['Capacity (MW)'] / total_grid_capacity) if 'Capacity (MW)' in substations.columns else (1 / len(substations))
hhi_index = (substations['Capacity Share'] ** 2).sum() * 10000

# Underserved Growth Regions (Few substations per geographic area)
region_counts = substations.groupby('Region')['Substation ID'].count().reset_index()
region_counts.columns = ['Region', 'Substation Count']
underserved_regions = region_counts.sort_values(by='Substation Count', ascending=True)

# ==========================================
# 4. TERMINAL EXECUTIVE REPORT
# ==========================================
print("==================================================")
print("       POWER GRID BUSINESS & RELIABILITY REPORT   ")
print("==================================================")
print(f"Top Asset Upgrade Candidates (>85% Load): {substations['Upgrade Candidate'].sum()} Substations")
print(f"High Fault-Risk Assets (>30 Years Old):   {substations['High Fault Risk'].sum()} Substations")
print(f"Grid Capacity Concentration Index (HHI):  {hhi_index:.2f} / 10000")
print("\n--- MAINTENANCE RATE BY UTILITY (%) ---")
print(maint_by_utility.round(2).to_string())
print("\n--- UNDERSERVED REGIONS (GROWTH OPPORTUNITIES) ---")
print(underserved_regions.head().to_string(index=False))
print("==================================================\n")

# ==========================================
# 5. GENERATE BUSINESS INTELLIGENCE DASHBOARD
# ==========================================
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Utility Footprint by Substation Count',
        'Substation Asset Age Distribution',
        'Capacity Utilization vs Asset Age (Risk Grid)',
        'Technical Line Loss Index (Highest Risk Transmission Runs)'
    )
)

# Plot 1: Footprint by Utility
footprint = substations.groupby('Name').size().reset_index(name='Count')
fig.add_trace(
    go.Bar(x=footprint['Name'], y=footprint['Count'], marker_color='teal', name='Substations'),
    row=1, col=1
)

# Plot 2: Asset Age Profile
fig.add_trace(
    go.Histogram(x=substations['Asset Age'], nbinsx=15, marker_color='indianred', name='Age Distribution'),
    row=1, col=2
)

# Plot 3: Capacity Utilization vs Asset Age
fig.add_trace(
    go.Scatter(
        x=substations['Asset Age'], 
        y=substations['Utilization (%)'], 
        mode='markers',
        marker=dict(size=10, color=substations['Upgrade Candidate'].map({True: 'red', False: 'blue'})),
        text=substations['Substation ID'],
        name='Substations'
    ),
    row=2, col=1
)

# Plot 4: Technical Loss Index Top Lines
top_loss_lines = lines.sort_values(by='Loss Proxy Index', ascending=False).head(10)
fig.add_trace(
    go.Bar(x=top_loss_lines['Line ID'].astype(str), y=top_loss_lines['Loss Proxy Index'], marker_color='orange', name='Loss Index'),
    row=2, col=2
)

fig.update_layout(
    title_text="National Grid Business Intelligence & Reliability Dashboard",
    height=800,
    showlegend=False,
    template="plotly_white"
)

# Save Interactive Dashboard
dashboard_file = "grid_business_intelligence_dashboard.html"
fig.write_html(dashboard_file)
print(f"Interactive BI Dashboard saved as '{dashboard_file}'. Open this file in your browser!")

## 9. Interactive Streamlit Dashboard (reference)

The full interactive dashboard (Overview / Network / Geography / Reliability / Search tabs, including an in-app N-1 contingency simulator) lives in `src/08_dashboard.py`. Streamlit apps are not meant to run inline in a notebook cell — launch it from a terminal instead:

```bash
streamlit run src/08_dashboard.py
```

The source is included below for completeness/reference.


In [ ]:
'''
"""
08_dashboard.py
National Electricity Grid Network Analysis — Task 3.1/3.2 (Week 3)
Interactive Streamlit dashboard pulling together EDA, network, geo, and BI results.

Run with:  streamlit run src/08_dashboard.py
"""

import streamlit as st
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from streamlit_folium import st_folium
import folium

st.set_page_config(page_title="Ghana National Grid Dashboard", layout="wide")

# ==========================================
# LOAD DATA (cached so it only loads once per session)
# ==========================================
@st.cache_data
def load_data():
    lines = pd.read_csv("data/lines.csv")
    substations = pd.read_csv("data/substations.csv")
    utilities = pd.read_csv("data/utilities.csv")
    master = pd.read_csv("data/masterDataset.csv")
    return lines, substations, utilities, master

lines, substations, utilities, master = load_data()

@st.cache_resource
def build_graph(_lines, _substations):
    G = nx.Graph()
    for _, row in _substations.iterrows():
        G.add_node(
            row["Substation ID"], 
            name=row.get("Name", f"Substation {row['Substation ID']}"),
            region=row.get("Region", "N/A"), 
            voltage=row.get("Voltage (kV)", 0),
            lat=row.get("Latitude", None),
            lon=row.get("Longitude", None)
        )
    for _, row in _lines.iterrows():
        G.add_edge(
            row["Source Substation ID"], 
            row["Destination Substation ID"],
            length=row.get("Length (km)", 1.0), 
            capacity=row.get("Capacity (MVA)", 1.0)
        )
    return G

G = build_graph(lines, substations)

st.title("Ghana National Grid Network Analysis Dashboard")

tab_overview, tab_network, tab_geo, tab_reliability, tab_search = st.tabs(
    ["Overview", "Network", "Geography", "Reliability", "Search"]
)

# ==========================================
# TAB 1 — OVERVIEW
# ==========================================
with tab_overview:
    st.subheader("Summary")

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Substations", len(substations))
    col2.metric("Transmission lines", len(lines))
    col3.metric("Utilities", len(utilities))
    col4.metric("Regions covered", substations["Region"].nunique())

    col5, col6, col7, col8 = st.columns(4)
    col5.metric("Active substations", (substations["Status"] == "Active").sum())
    col6.metric("Total line length (km)", f"{lines['Length (km)'].sum():,.0f}")
    col7.metric("Avg. substation capacity (MVA)", f"{substations['Capacity (MVA)'].mean():.1f}")
    
    st.markdown("---")
    c1, c2 = st.columns(2)
    with c1:
        st.plotly_chart(
            px.bar(substations["Region"].value_counts().reset_index(),
                   x="Region", y="count", title="Substations by region"),
            use_container_width=True
        
        )
    with c2:
        st.plotly_chart(
            px.pie(substations, names="Voltage (kV)", title="Voltage tier distribution"),
            use_container_width=True
        )

# ==========================================
# TAB 2 — NETWORK
# ==========================================
with tab_network:
    st.subheader("Network structure and vulnerability")

    region_filter = st.selectbox("Filter by region", ["All"] + sorted(substations["Region"].dropna().unique().tolist()))

    degree_cent = nx.degree_centrality(G)
    betweenness_cent = nx.betweenness_centrality(G, weight="length")
    articulation_points = list(nx.articulation_points(G))

    centrality_df = pd.DataFrame({
        "Substation ID": list(G.nodes()),
        "Name": [G.nodes[n]["name"] for n in G.nodes()],
        "Region": [G.nodes[n]["region"] for n in G.nodes()],
        "Degree Centrality": [degree_cent[n] for n in G.nodes()],
        "Betweenness Centrality": [betweenness_cent[n] for n in G.nodes()],
        "Critical Node": [n in articulation_points for n in G.nodes()],
    }).sort_values("Degree Centrality", ascending=False)

    if region_filter != "All":
        centrality_df = centrality_df[centrality_df["Region"] == region_filter]
    else:
        filtered_centrality = centrality_df

    st.dataframe(filtered_centrality, use_container_width=True)

    st.markdown("---")
    st.markdown("#### N-1 contingency test")
    top_hub_id = centrality_df.iloc[0]["Substation ID"] if not centrality_df.empty else None
    if top_hub_id is not None and st.button(f"Simulate removing top hub ({centrality_df.iloc[0]['Name']})"):
        G_test = G.copy()
        G_test.remove_node(top_hub_id)
        if nx.is_connected(G_test):
            st.success("Network remains fully connected after removing this substation.")
        else:
            comps = list(nx.connected_components(G_test))
            st.error(f"Network fragments into {len(comps)} disconnected pieces.")
            for i, comp in enumerate(comps, 1):
                names = [G.nodes[n]["name"] for n in comp]
                st.write(f"Group {i} ({len(comp)} substations): {', '.join(names)}")

# ==========================================
# TAB 3 — GEOGRAPHY
# ==========================================
with tab_geo:
    st.subheader("Geographic distribution")

    voltage_options = sorted(substations["Voltage (kV)"].dropna().unique().tolist())
    selected_voltages = st.multiselect("Filter by voltage (kV)", voltage_options, default=voltage_options)

    filtered_subs = substations[substations["Voltage (kV)"].isin(selected_voltages)]
    valid_sub_ids = set(filtered_subs["Substation ID"])
    
    m = folium.Map(location=[7.9465, -1.0232], zoom_start=6, tiles="CartoDB Positron")
    voltage_colors = {11: "purple", 33: "green", 69: "blue", 161: "orange", 330: "red"}

    sub_coords = filtered_subs.set_index("Substation ID")[["Latitude", "Longitude"]].to_dict("index")
    for _, line in lines.iterrows():
        src = line["Source Substation ID"]
        dst = line["Destination Substation ID"]
        if src in sub_coords and dst in sub_coords:
            p1 = [sub_coords[src]["Latitude"], sub_coords[src]["Longitude"]]
            p2 = [sub_coords[dst]["Latitude"], sub_coords[dst]["Longitude"]]
            folium.PolyLine(
                locations=[p1, p2],
                color="#555555",
                weight=1.5,
                opacity=0.6,
                tooltip=f"Line: {src} ↔ {dst} ({line.get('Length (km)', 'N/A')} km)"
            ).add_to(m)
            
    for _, row in filtered_subs.dropna(subset=["Latitude", "Longitude"]).iterrows():
        v = row.get("Voltage (kV)", 0)
        folium.CircleMarker(
            location=[row["Latitude"], row["Longitude"]],
            radius=5 + (v / 50),
            color=voltage_colors.get(v, "gray"),
            fill=True,
            fill_opacity=0.7,
            tooltip=f"{row.get('Name', 'Unknown')} ({v}kV) - {row.get('Region', '')}",
        ).add_to(m)

    st_folium(m, width=None, height=550, use_container_width=True)

# ==========================================
# TAB 4 — RELIABILITY / BI
# ==========================================
with tab_reliability:
    st.subheader("Business Intelligence and Reliability Analysis")

    c1, c2 = st.columns(2)
    with c1:
        if "Status" in substations.columns:
            status_counts = (
                substations["Status"]
                .value_counts()
                .rename_axis("Status")
                .reset_index(name="count")
            )
            st.plotly_chart(px.bar(status_counts, x="Status", y="count", title="Substation Operational Status"), use_container_width=True)
    with c2:
        if "Commissioning Year" in substations.columns:
            st.plotly_chart(
                px.histogram(substations, x="Commissioning Year", title="Substation Age Distribution (Commissioning Year)"),
                use_container_width=True
            )

    st.info("💡 Capacity utilization and asset-age degradation models can be integrated here upon Task 7 completion.")

# ==========================================
# TAB 5 — SEARCH
# ==========================================
with tab_search:
    st.subheader("Substation finder")

    search_term = st.text_input("Search by substation name")
    if search_term:
        results = substations[substations["Name"].str.contains(search_term, case=False, na=False)]
        st.dataframe(results, use_container_width=True)

    st.markdown("---")
    st.subheader("Utility comparison")
    selected_utilities = st.multiselect("Select utilities to compare", utilities["Name"].tolist())
    if selected_utilities:
        ids = utilities[utilities["Name"].isin(selected_utilities)]["Utility ID"]
        comp = lines[lines["Utility ID"].isin(ids)]
        st.plotly_chart(
            px.bar(comp["Utility ID"].value_counts().reset_index(), x="Utility ID", y="count",
                   title="Lines operated per selected utility"),
            use_container_width=True
        )
'''
# ^ Reference only — run via `streamlit run src/08_dashboard.py`, not as a notebook cell.